# LASSO, OLS, and Random Forest: Variable Selection and Model Comparison
This notebook extends the log-log OLS from `ols_regression.ipynb` by:
1. **LASSO** — uses a penalized regression over a broad feature set (83 variables) to identify which predictors survive shrinkage
2. **OLS vs. LASSO comparison** — checks whether both models agree on sign and significance for the 5 shared predictors
3. **Model performance** — compares in-sample R², out-of-sample R², and RMSE across Log FE OLS / OLS (full) / LASSO
4. **Random Forest robustness check** — non-parametric model used to verify LASSO's variable importance ranking

Reads from: `data/college_scorecard_clean.csv`  
Writes to: `output/figures/`

---
## 1. Load Data and Construct Features

Log-transform skewed continuous variables before feeding into LASSO.  
All categorical variables are one-hot encoded; `drop_first=True` avoids perfect multicollinearity (the dummy variable trap).

In [ ]:
import pandas as pd
import numpy as np

input_path = "../data/college_scorecard_clean.csv"
df_raw = pd.read_csv(input_path)

# Log-transform skewed continuous variables 
# Log transforms reduce the influence of outliers and make relationships more linear
df_raw['log_earnings']     = np.log(df_raw['md_earn_10yr'])
df_raw['log_adm_rate']     = np.log(df_raw['adm_rate'])
df_raw['log_tuition']      = np.log(df_raw['tuition'])
df_raw['log_net_cost']     = np.log(df_raw['net_cost'])
df_raw['log_instruct_exp'] = np.log(df_raw['instruct_exp_fte'])
df_raw['log_avg_fac_sal']  = np.log(df_raw['avg_fac_sal'])
df_raw['log_enrollment']   = np.log(df_raw['enrollment'])

# One-hot encode categorical variables 
# drop_first=True drops one category per variable to avoid the dummy variable
# trap (perfect collinearity) 
# Ex: control drops 'private_fp', so coefficients
# on private_np and public are relative to for-profit schools
state_dummies    = pd.get_dummies(df_raw['state'],    prefix='state',    drop_first=True)
control_dummies  = pd.get_dummies(df_raw['control'],  prefix='control',  drop_first=True)
pred_deg_dummies = pd.get_dummies(df_raw['pred_deg'], prefix='pred_deg', drop_first=True)
region_dummies   = pd.get_dummies(df_raw['region'],   prefix='region',   drop_first=True)

# Assemble feature matrix
continuous_vars = [
    'log_adm_rate', 'log_tuition', 'log_net_cost', 'log_instruct_exp',
    'log_avg_fac_sal', 'log_enrollment',
    'pct_pell', 'pct_ft_fac', 'med_fam_inc', 'pct_first_gen',
    'hbcu', 'hsi',
]

X_lasso = pd.concat([
    df_raw[continuous_vars],
    control_dummies,
    pred_deg_dummies,
    region_dummies,
    state_dummies        # included so LASSO can absorb state-level variation
], axis=1)

y_lasso = df_raw['log_earnings']

# Drop any row where at least one feature or the outcome is missing
# Using a boolean mask (rather than dropna on the full df) so X and y stay aligned
mask    = X_lasso.notna().all(axis=1) & y_lasso.notna()
X_lasso = X_lasso[mask]
y_lasso = y_lasso[mask]

print(f"Observations: {len(y_lasso)}")
print(f"Features:     {X_lasso.shape[1]}")
print(f"\nContinuous features going into LASSO:")
for v in continuous_vars:
    print(f"  {v}")

---
## 2. Standardize and Fit LASSO with Cross-Validated Penalty

**Why standardize?**  
LASSO penalizes the *sum of absolute coefficients*. Without standardization, variables measured in large units (e.g. dollars) would be penalized more aggressively than variables in small units (e.g. rates), making the penalty unfair. `StandardScaler` puts every feature on a mean-0, SD-1 scale so the penalty applies equally.

**Choosing the penalty (α):**  
`LassoCV` fits LASSO across 100 values of α and selects the one that minimizes 5-fold cross-validation error. Higher α = stronger shrinkage = more variables zeroed out.

In [ ]:
from sklearn.linear_model import LassoCV
from sklearn.preprocessing import StandardScaler

# Standardization
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X_lasso)

# LASSO Model
lasso_cv = LassoCV(
    cv           = 5,         # 5-fold cross-validation to select alpha
    max_iter     = 10000,     # LASSO can be slow to converge; raise iter limit to ensure convergence
    n_alphas     = 100,       # search grid of 100 alpha values on a log scale
    random_state = 42
)
lasso_cv.fit(X_scaled, y_lasso)

print(f"Best alpha (penalty strength): {lasso_cv.alpha_:.6f}")
print(f"R-squared on training data:    {lasso_cv.score(X_scaled, y_lasso):.4f}")

n_kept = (lasso_cv.coef_ != 0).sum()
print(f"\nTotal features:       {X_lasso.shape[1]}")
print(f"Variables kept:       {n_kept}")
print(f"Variables zeroed out: {X_lasso.shape[1] - n_kept}")

---
## 3. LASSO Results

Coefficients on standardized inputs — magnitudes reflect relative importance within this model.  
State FE results are summarized at the bottom (suppressed from the main table for readability).

In [ ]:
# Build a results dataframe sorted by absolute coefficient size
coef_df = pd.DataFrame({
    'variable':   X_lasso.columns,
    'lasso_coef': lasso_cv.coef_
}).sort_values('lasso_coef', key=abs, ascending=False)

# Separate substantive variables from state fixed effects for display
main_vars  = coef_df[~coef_df['variable'].str.startswith('state_')].copy()
state_vars = coef_df[coef_df['variable'].str.startswith('state_')].copy()

print("="*55)
print("LASSO RESULTS — Main Variables Only")
print(f"Alpha = {lasso_cv.alpha_:.6f} | R² = {lasso_cv.score(X_scaled, y_lasso):.4f}")
print("="*55)
print(f"{'Variable':<25} {'Coef':>10}  {'Status'}")
print("-"*55)
for _, row in main_vars.iterrows():
    status = "✓ kept" if row['lasso_coef'] != 0 else "✗ zeroed out"
    print(f"{row['variable']:<25} {row['lasso_coef']:>10.4f}  {status}")
print("-"*55)
print(f"\nState FEs kept: {(state_vars['lasso_coef'] != 0).sum()} / {len(state_vars)}")

---
## 4. Visualization: LASSO Coefficient Plot

Plots the 15 substantive predictors (no state/region dummies) ranked by coefficient magnitude.  
Blue = variable survived shrinkage; gray = zeroed out by LASSO.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Subset to the 15 substantive predictors for the plot 
# state and region dummies are controlled for but not the focus of the analysis
focus_vars = [
    'log_adm_rate', 'log_tuition', 'log_net_cost', 'log_instruct_exp',
    'log_avg_fac_sal', 'log_enrollment', 'pct_pell', 'pct_ft_fac',
    'med_fam_inc', 'pct_first_gen', 'hbcu', 'hsi',
    'control_private_np', 'control_public', 'pred_deg_1'
]

focus_df = coef_df[coef_df['variable'].isin(focus_vars)].copy()
focus_df = focus_df.sort_values('lasso_coef', ascending=True)

# Color bars by whether the variable survived LASSO shrinkage
colors = ['#2196F3' if v != 0 else '#CCCCCC' for v in focus_df['lasso_coef']]

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(focus_df['variable'], focus_df['lasso_coef'],
        color=colors, edgecolor='black', linewidth=0.5)
ax.axvline(0, color='black', linewidth=0.8)  # zero line: right = positive effect on earnings

ax.set_xlabel('LASSO Coefficient (standardized inputs)', fontsize=11)
ax.set_title(
    'LASSO Variable Selection: Predictors of Graduate Earnings\n'
    f'College Scorecard Data | State FEs included | α = {lasso_cv.alpha_:.6f}',
    fontsize=11, fontweight='bold'
)

kept_patch   = mpatches.Patch(color='#2196F3', label='Kept by LASSO')
zeroed_patch = mpatches.Patch(color='#CCCCCC', label='Zeroed out by LASSO')
ax.legend(handles=[kept_patch, zeroed_patch], loc='lower right')

# Annotate zeroed-out bars so it's clear they're EXACTLY zero
for _, row in focus_df.iterrows():
    if row['lasso_coef'] == 0:
        ax.text(0.001, list(focus_df['variable']).index(row['variable']),
                ' zeroed out', va='center', color='gray', fontsize=8)

plt.tight_layout()
plt.savefig('../output/figures/lasso_coefficients.png', dpi=150, bbox_inches='tight')
plt.show()

---
## 5. OLS vs. LASSO Comparison

Checks whether the log-log OLS (Notebook 2) and LASSO agree on **direction** for the 5 shared predictors.  
Note: magnitudes are not directly comparable — OLS coefficients are on the original log scale while LASSO coefficients are on the standardized scale.

OLS results below are hardcoded from Notebook 2 output. Re-run Notebook 2 and update these values if the sample changes.

In [ ]:
# OLS coefficients and p-values copied from Notebook 2 (log-log FE model).
# Hardcoded here to avoid re-running the full OLS — update if sample changes.
ols_results = {
    'log_adm_rate':   {'coef': -0.064, 'pval': 0.012},
    'sat_composite':  {'coef':  0.000, 'pval': 0.000},
    'log_tuition':    {'coef':  0.092, 'pval': 0.027},
    'pct_pell':       {'coef': -0.332, 'pval': 0.058},
    'control_public': {'coef':  0.030, 'pval': 0.163},
}

# Build a lookup dict so we can pull LASSO coefficients by variable name
lasso_coef_lookup = dict(zip(coef_df['variable'], coef_df['lasso_coef']))

print("="*72)
print(f"{'Variable':<22} {'OLS Coef':>10} {'OLS Sig':>8} {'LASSO Coef':>12} {'Agreement'}")
print("="*72)

for var, vals in ols_results.items():
    ols_c = vals['coef']
    ols_p = vals['pval']
    las_c = lasso_coef_lookup.get(var, 0.0)

    stars = ("***" if ols_p < 0.01 else "**" if ols_p < 0.05 else "*" if ols_p < 0.1 else "")

    # Agreement = same sign in both models; "LASSO drops" = zeroed out entirely
    if las_c == 0:
        agreement = "LASSO drops"
    elif (ols_c > 0) == (las_c > 0):
        agreement = "✓ agree"
    else:
        agreement = "✗ disagree"

    print(f"{var:<22} {str(ols_c):>10} {stars:>8} {las_c:>12.4f} {agreement}")

print("="*72)
print("\nNote: OLS coefficients are on original log scale.")
print("LASSO coefficients are on standardized scale.")
print("Comparison is directional only, not magnitude.")

# Variables in the LASSO feature set that OLS did not include 
# these are candidates for omitted variable bias in the OLS specification.
print("\n--- Variables LASSO found that OLS didn't include ---")
lasso_only = [
    'log_avg_fac_sal', 'med_fam_inc', 'pct_first_gen',
    'hbcu', 'hsi', 'log_net_cost', 'log_instruct_exp',
    'log_enrollment', 'pct_ft_fac'
]
for var in lasso_only:
    c      = lasso_coef_lookup.get(var, 0.0)
    status = f"{c:.4f}  ✓ kept" if c != 0 else "0.0000  ✗ zeroed out"
    print(f"  {var:<22} {status}")

---
## 6. Model Performance Comparison

Compares three models on the same prediction task (log earnings):
- **Log FE OLS** — 5 predictors + state FEs (Notebook 2 specification)
- **OLS full** — all 83 features, no penalty (overfitting benchmark)
- **LASSO** — 83 features with cross-validated L1 penalty

The key diagnostic is the **overfitting gap**: how much R² drops from in-sample to out-of-sample.  
A large gap means the model memorized the training data rather than learning generalizable patterns.

In [ ]:
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression, Lasso
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import r2_score, mean_squared_error
import statsmodels.formula.api as smf

# Log FE OLS (re-estimated for out-of-sample evaluation) 
# Re-load data so this cell can run independently of Notebook 2
df_ols = pd.read_csv(input_path)
df_ols['log_earnings'] = np.log(df_ols['md_earn_10yr'])
df_ols['log_adm_rate'] = np.log(df_ols['adm_rate'])
df_ols['log_tuition']  = np.log(df_ols['tuition'])

formula = ("log_earnings ~ log_adm_rate + sat_composite + log_tuition "
           "+ C(control) + pct_pell + C(state)")

keep   = ['log_earnings', 'log_adm_rate', 'sat_composite',
          'log_tuition', 'control', 'pct_pell', 'state']
df_ols = df_ols[keep].replace([np.inf, -np.inf], np.nan).dropna()

ols_fe        = smf.ols(formula=formula, data=df_ols).fit(cov_type='HC3')
ols_fe_r2_in  = ols_fe.rsquared
ols_fe_rmse_in = np.sqrt(ols_fe.mse_resid)

# Manual 80/20 train-test split for the FE OLS 
np.random.seed(42)
idx = np.random.permutation(len(df_ols))
cut = int(0.8 * len(idx))
tr, te = idx[:cut], idx[cut:]

ols_fit_tr      = smf.ols(formula=formula, data=df_ols.iloc[tr]).fit(cov_type='HC3')
y_hat_te        = ols_fit_tr.predict(df_ols.iloc[te])
y_te            = df_ols['log_earnings'].iloc[te]

ols_fe_r2_out   = r2_score(y_te, y_hat_te)
ols_fe_rmse_out = np.sqrt(mean_squared_error(y_te, y_hat_te))

# OLS full (sklearn) and LASSO 
# shared 80/20 split on X_scaled --> using the same split for both ensures their test sets are identical,
# making RMSE and R^2 directly comparable between them
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_lasso, test_size=0.2, random_state=42
)

ols_full = LinearRegression().fit(X_train, y_train)

ols_full_r2_in    = r2_score(y_train, ols_full.predict(X_train))
ols_full_r2_out   = r2_score(y_test,  ols_full.predict(X_test))
ols_full_rmse_in  = np.sqrt(mean_squared_error(y_train, ols_full.predict(X_train)))
ols_full_rmse_out = np.sqrt(mean_squared_error(y_test,  ols_full.predict(X_test)))
# 5-fold CV R^2 as an additional generalization check
ols_full_cv       = cross_val_score(LinearRegression(), X_scaled, y_lasso, cv=5, scoring='r2').mean()

lasso_m = Lasso(alpha=lasso_cv.alpha_, max_iter=10000).fit(X_train, y_train)

lasso_r2_in    = r2_score(y_train, lasso_m.predict(X_train))
lasso_r2_out   = r2_score(y_test,  lasso_m.predict(X_test))
lasso_rmse_in  = np.sqrt(mean_squared_error(y_train, lasso_m.predict(X_train)))
lasso_rmse_out = np.sqrt(mean_squared_error(y_test,  lasso_m.predict(X_test)))
lasso_cv_r2    = cross_val_score(
    Lasso(alpha=lasso_cv.alpha_, max_iter=10000), X_scaled, y_lasso, cv=5, scoring='r2'
).mean()

# Summary table 
print("="*72)
print(f"{'Metric':<28} {'Log FE OLS':>13} {'OLS 83 vars':>13} {'LASSO':>13}")
print("="*72)
print(f"{'N (estimation sample)':<28} {'1,163':>13} {'1,483':>13} {'1,483':>13}")
print(f"{'Features':<28} {'5 + FEs':>13} {'83':>13} {'83→40':>13}")
print(f"{'R² in-sample':<28} {ols_fe_r2_in:>13.4f} {ols_full_r2_in:>13.4f} {lasso_r2_in:>13.4f}")
print(f"{'R² out-of-sample':<28} {ols_fe_r2_out:>13.4f} {ols_full_r2_out:>13.4f} {lasso_r2_out:>13.4f}")
print(f"{'CV R² (5-fold)':<28} {'N/A':>13} {ols_full_cv:>13.4f} {lasso_cv_r2:>13.4f}")
print(f"{'RMSE in-sample':<28} {ols_fe_rmse_in:>13.4f} {ols_full_rmse_in:>13.4f} {lasso_rmse_in:>13.4f}")
print(f"{'RMSE out-of-sample':<28} {ols_fe_rmse_out:>13.4f} {ols_full_rmse_out:>13.4f} {lasso_rmse_out:>13.4f}")
print(f"{'Overfitting gap (R²)':<28} {ols_fe_r2_in-ols_fe_r2_out:>13.4f} {ols_full_r2_in-ols_full_r2_out:>13.4f} {lasso_r2_in-lasso_r2_out:>13.4f}")
print("="*72)
print("Note: Log FE OLS uses a smaller sample (requires non-missing SAT scores).")
print("OLS full and LASSO use identical samples and features for a fair comparison.")

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

models   = ['Log FE OLS\n(N=1,163)', 'OLS 83 vars\n(N=1,483)', 'LASSO\n(N=1,483)']
r2_in    = [ols_fe_r2_in,    ols_full_r2_in,    lasso_r2_in]
r2_out   = [ols_fe_r2_out,   ols_full_r2_out,   lasso_r2_out]
rmse_out = [ols_fe_rmse_out, ols_full_rmse_out, lasso_rmse_out]
colors   = ['steelblue', 'tomato', 'seagreen']

x, w = np.arange(len(models)), 0.35

# Grouped bar chart: solid = in-sample, faded = out-of-sample
# A large drop from solid to faded bar signals overfitting.
axes[0].bar(x - w/2, r2_in,  w, label='In-sample',     color=colors, alpha=0.9, edgecolor='black')
axes[0].bar(x + w/2, r2_out, w, label='Out-of-sample', color=colors, alpha=0.4, edgecolor='black')
axes[0].set_xticks(x)
axes[0].set_xticklabels(models, fontsize=9)
axes[0].set_ylabel('R²')
axes[0].set_ylim(0, 1)
axes[0].set_title('R²: In-sample vs Out-of-sample')
axes[0].legend(['In-sample', 'Out-of-sample'])

# Annotate overfitting gap above each pair of bars
for i, (ri, ro) in enumerate(zip(r2_in, r2_out)):
    axes[0].annotate(f'gap={ri-ro:.3f}',
                     xy=(x[i], max(ri, ro) + 0.02),
                     ha='center', fontsize=8, color='black')

# Out-of-sample RMSE only (matters most for generalization)
bars = axes[1].bar(models, rmse_out, color=colors, edgecolor='black', alpha=0.85)
axes[1].set_ylabel('RMSE (out-of-sample)')
axes[1].set_title('Out-of-sample RMSE\n(lower is better)')
for bar, val in zip(bars, rmse_out):
    axes[1].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 0.002,
                 f'{val:.4f}', ha='center', fontsize=9)

plt.suptitle('Model Performance Comparison: Log FE OLS vs OLS vs LASSO',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('../output/figures/model_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

#### Interpreting RMSE in Dollar Terms

RMSE is in log-earnings units. To communicate the practical magnitude of prediction error, we convert back to approximate dollar terms.

In [ ]:
# RMSE from the LASSO out-of-sample evaluation above (in log-earnings units)
rmse_log     = lasso_rmse_out
avg_earnings = np.exp(y_lasso.mean())

print(f"Mean log earnings:          {y_lasso.mean():.4f}")
print(f"Mean earnings (level):      ${avg_earnings:,.0f}")
print(f"RMSE in log points:         {rmse_log:.4f}")
# Approximation: for small log errors, exp(rmse) - 1 = rmse, so rmse = % error
print(f"Approx % prediction error:  {rmse_log*100:.1f}%")
print(f"In dollar terms (~):        ${avg_earnings * rmse_log:,.0f}")

---
## 7. Random Forest Robustness Check

A Random Forest is a non-parametric, non-linear ensemble model.  
If RF and LASSO produce similar variable importance rankings, it strengthens confidence that those predictors are genuinely informative.

**RF hyperparameters:**
- `n_estimators=500`: 500 trees; more trees reduce variance in importance estimates
- `min_samples_leaf=5`: each leaf must have ≥5 observations —-> limits overfitting on small branches
- `max_depth=None`: trees grow fully (depth controlled by `min_samples_leaf` instead)
- `n_jobs=-1`: use all available CPU cores to parallelize tree fitting

In [ ]:
from sklearn.ensemble import RandomForestRegressor

# Reuse the same X_train / X_test split from the model comparison cell
rf = RandomForestRegressor(
    n_estimators    = 500,
    max_depth       = None,
    min_samples_leaf= 5,     # prevents overfitting to tiny leaf nodes
    random_state    = 42,
    n_jobs          = -1     # parallelize across all CPU cores
)
rf.fit(X_train, y_train)

rf_r2_in  = rf.score(X_train, y_train)
rf_r2_out = rf.score(X_test,  y_test)
rf_rmse   = np.sqrt(mean_squared_error(y_test, rf.predict(X_test)))

print(f"RF R² in-sample:     {rf_r2_in:.4f}")
print(f"RF R² out-of-sample: {rf_r2_out:.4f}")
print(f"RF RMSE (test):      {rf_rmse:.4f}")
print(f"RF overfitting gap:  {rf_r2_in - rf_r2_out:.4f}  (RF typically overfits more than LASSO)")

# Extract importance for the 15 substantive variables
# RF importance = mean decrease in impurity across all trees.
# We restrict to the same focus_vars as the LASSO plot for a side-by-side comparison.
focus_vars = [
    'log_adm_rate', 'log_tuition', 'log_net_cost', 'log_instruct_exp',
    'log_avg_fac_sal', 'log_enrollment', 'pct_pell', 'pct_ft_fac',
    'med_fam_inc', 'pct_first_gen', 'hbcu', 'hsi',
    'control_private_np', 'control_public', 'pred_deg_1'
]

focus_idx   = [list(X_lasso.columns).index(v) for v in focus_vars if v in X_lasso.columns]
focus_names = [X_lasso.columns[i] for i in focus_idx]

rf_importance = pd.DataFrame({
    'variable':      focus_names,
    'rf_importance': rf.feature_importances_[focus_idx]
}).sort_values('rf_importance', ascending=True)

lasso_abs = (coef_df[coef_df['variable'].isin(focus_vars)]
             .assign(lasso_abs=lambda d: d['lasso_coef'].abs())
             [['variable', 'lasso_abs']])

# Merge so each row has both RF importance and LASSO |coef| for the same variable
compare = rf_importance.merge(lasso_abs, on='variable').sort_values('rf_importance', ascending=True)

# Side-by-side bar chart 
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

axes[0].barh(compare['variable'], compare['rf_importance'],
             color='#2ecc71', edgecolor='black', linewidth=0.5)
axes[0].set_title('Random Forest', fontweight='bold')
axes[0].set_xlabel('Importance Score (higher = more predictive)')

# Color LASSO bars by whether the variable survived shrinkage (same scheme as earlier plot)
colors_lasso = ['#2196F3' if v != 0 else '#cccccc' for v in compare['lasso_abs']]
axes[1].barh(compare['variable'], compare['lasso_abs'],
             color=colors_lasso, edgecolor='black', linewidth=0.5)
axes[1].set_title('LASSO', fontweight='bold')
axes[1].set_xlabel('|Coefficient| (higher = more predictive)')

plt.suptitle('Robustness Check: Do RF and LASSO Agree on the Strongest Predictors?',
             fontweight='bold', fontsize=11)
plt.tight_layout()
plt.savefig('../output/figures/rf_vs_lasso.png', dpi=150, bbox_inches='tight')
plt.show()

# Rank correlation table 
# If RF rank and LASSO rank are similar, both models tell the same story about
# which institutional characteristics matter most for graduate earnings.
compare['rf_rank']    = compare['rf_importance'].rank(ascending=False).astype(int)
compare['lasso_rank'] = compare['lasso_abs'].rank(ascending=False).astype(int)
compare = compare.sort_values('rf_rank')

print("\n" + "="*55)
print(f"{'Variable':<25} {'RF Rank':>8} {'LASSO Rank':>10}")
print("="*55)
for _, row in compare.iterrows():
    print(f"{row['variable']:<25} {row['rf_rank']:>8} {row['lasso_rank']:>10}")
print("="*55)